# Ecommerce Dataset Batch Table Generation

Creats the batch product dataset and product aggregations datasets.

In [0]:
import os
import yaml
from datetime import date

import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Load project config
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
TABLE = f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}"
STREAMING_BACKFILL_TABLE = f"{CATALOG}.{SCHEMA}.{config['stream_backfill_table']}"
PRODUCT_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_table']}"
PRODUCT_AGGREGATE_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_aggregates_table']}"

print(f"Clickstream raw source table: {TABLE}")
print(f"Streaming backfille table: {STREAMING_BACKFILL_TABLE}\n")
print(f"Product table: {PRODUCT_TABLE}")
print(f"Product aggregate table: {PRODUCT_AGGREGATE_TABLE}")

#### Product table

In [0]:
# ---------------------------------------------------------------------------
# Product catalog table — one row per product per day.
#
# The raw event table has ~110M rows, but most rows for the same product
# carry identical attributes. Deduplicating to the last observation per
# product per day:
#   - Handles attribute drift (price changes, recategorization, brand updates)
#   - Preserves event_time for point-in-time ColumnSelection joins
#   - Aligns with SlidingWindow(slide=1d) granularity
#   - Includes event_time_copy for item_recency (Max aggregation)
#
# event_time is shifted by DELTA_DAYS to align with the ingestion table
# timestamps (02_build_backfill shifts 2019-11-30 → D=2026-09-25).
# All dates (Oct 1 – Nov 30) are included.
# ---------------------------------------------------------------------------

final_df = (spark.table(TABLE)
            .withColumn("_cat", F.coalesce(F.col("category_code"), F.lit("unknown")))
            .withColumn("_cat_parts", F.split("_cat", "\\."))
            .withColumn("category_l1", F.coalesce(F.expr("get(_cat_parts, 0)"), F.lit("unknown")))
            .withColumn("category_l2", F.coalesce(F.expr("get(_cat_parts, 1)"), F.lit("unknown")))
            .withColumn("category_l3", F.coalesce(F.expr("get(_cat_parts, 2)"), F.lit("unknown")))
            .withColumn("brand", F.coalesce(F.col("brand"), F.lit("unknown")))
            .withColumn("log_price", F.log1p(F.coalesce(F.col("price"), F.lit(0))))
            .withColumn("event_time_copy", F.col("event_time"))
            .drop("_cat", "_cat_parts"))

DELTA_DAYS = config["delta_days"]

w_product_daily = Window.partitionBy(
    "product_id", F.to_date("event_time")
).orderBy(F.desc("event_time"))

product_df = (
    final_df
    .withColumn("event_time", F.col("event_time") + F.expr(f"INTERVAL {DELTA_DAYS} DAYS"))
    .withColumn("_rn", F.row_number().over(w_product_daily))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("log_price", F.log1p(F.coalesce(F.col("price"), F.lit(0))))
    .select(
        "product_id", "event_time",
        "price", "log_price", "brand",
        "category_l1", "category_l2", "category_l3",
        "category_id", "category_code",
    )
)

product_df.write.mode("overwrite").saveAsTable(PRODUCT_TABLE)
product_df = spark.table(PRODUCT_TABLE)

print(f"Full events:     {final_df.count():,}")
print(f"Product catalog: {product_df.count():,} (1 row per product per day)")
print(f"Distinct items:  {product_df.select('product_id').distinct().count():,}")
display(product_df.limit(10))

Product aggregate tables

In [0]:
backfille_records = spark.table(STREAMING_BACKFILL_TABLE)

df_flat = (backfille_records.selectExpr(
                    "value.user_session as user_session",
                    "value.user_id as user_id",
                    "value.event_time as event_time",
                    "value.event_time as event_time_copy",
                    "value.event_type as event_type",
                    "value.product_id as product_id",
                    "value.category_id as category_id",
                    "value.category_code as category_code",
                    "value.brand as brand",
                    "value.price as price",
                ))

df_flat.write.mode("overwrite").format("delta").saveAsTable(PRODUCT_AGGREGATE_TABLE)

display(spark.table(PRODUCT_AGGREGATE_TABLE))